# Пайплайн аугментации

Ноутбук последовательно:
1. Разбивает данные на **train/test** (80/20, стратифицированно)
2. Аугментирует **только train** через 3 этапа
3. Оценивает baseline-классификаторы **на test**

**Этапы аугментации:**
1. LLM-генерация (< 15 → 15)
2. ruT5-парафраз с чанкованием (< 35 → 35)
3. NLLB back-translation с чанкованием (< 50 → 50)

**Классификация (baseline):**
- Linear SVM (TF-IDF)
- Logistic Regression (TF-IDF)
- Multinomial Naive Bayes (TF-IDF)
- rubert-tiny2 (fine-tuning)

___
## Подготовка окружения (Запуск в Colab)

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/KVTur23/Mifi_VKR.git'
REPO_BRANCH = 'rut5-nllb-chunked-augment'
REPO_DIR = Path('/content/VKR')
PROJECT_ROOT = REPO_DIR / 'code'

DRIVE_ROOT = Path('/content/drive/MyDrive/VKR')
DRIVE_DATA = DRIVE_ROOT / 'Data'
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_LOGS = DRIVE_ROOT / 'logs'


In [3]:
import shutil
import subprocess

if (REPO_DIR / '.git').exists():
    print('Репо уже клонирован — обновляю из origin')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)
else:
    print(f'Клонирую {REPO_URL} в {REPO_DIR}')
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

def link_to_drive(name, drive_path):
    local_path = PROJECT_ROOT / name
    drive_path.parent.mkdir(parents=True, exist_ok=True)

    drive_missing_or_empty = not drive_path.exists() or not any(drive_path.iterdir())
    if drive_missing_or_empty:
        if local_path.exists() and not local_path.is_symlink():
            if drive_path.exists():
                shutil.rmtree(drive_path)
            shutil.copytree(local_path, drive_path)
        else:
            drive_path.mkdir(parents=True, exist_ok=True)

    if local_path.is_symlink() or local_path.is_file():
        local_path.unlink()
    elif local_path.exists():
        shutil.rmtree(local_path)

    local_path.symlink_to(drive_path, target_is_directory=True)
    print(f'{name}: {local_path} -> {drive_path}')

link_to_drive('Data', DRIVE_DATA)
link_to_drive('results', DRIVE_RESULTS)
link_to_drive('logs', DRIVE_LOGS)

commit = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'],
    capture_output=True, text=True, check=True,
).stdout.strip()
print(f'\nТекущий коммит: {commit}')


Репо уже клонирован — обновляю из origin
Data: /content/VKR/code/Data -> /content/drive/MyDrive/VKR/Data
results: /content/VKR/code/results -> /content/drive/MyDrive/VKR/results
logs: /content/VKR/code/logs -> /content/drive/MyDrive/VKR/logs

Текущий коммит: 8d3e2d4 best_results


In [4]:
import sys
import pandas as pd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%cd {PROJECT_ROOT}

from src.utils.data_loader import (
    load_dataset, get_class_distribution, split_train_test, load_test_set,
    LABEL_COL, RANDOM_SEED, DATA_DIR, ORIGINAL_FILE,
)

print(f'Корень проекта: {PROJECT_ROOT}')
print(f'Папка данных:   {DATA_DIR}')
print(f'Drive Data:     {DRIVE_DATA}')

from src.utils.pipeline_config import load_pipeline_config

# === GPU ПРОФИЛЬ — МЕНЯЙ ЗДЕСЬ ===
GPU = 'A100_40'  # варианты: 'T4', 'L4', 'A100_40', 'A100_80', 'H100'
pipeline_cfg = load_pipeline_config(GPU)


/content/VKR/code
Корень проекта: /content/VKR/code
Папка данных:   /content/VKR/code/Data
Drive Data:     /content/drive/MyDrive/VKR/Data
[Config] GPU: A100_40 (40GB), NLLB: facebook/nllb-200-3.3B, batch: 64


## Подготовка окружения (Локальный запуск)

In [5]:
# import sys
# from pathlib import Path
# import pandas as pd

# PROJECT_ROOT = Path("/Users/kvt/Documents/VKR/code")         

# # Добавляем корень в sys.path
# if str(PROJECT_ROOT) not in sys.path:
#     sys.path.insert(0, str(PROJECT_ROOT))

# %cd {PROJECT_ROOT}

# from src.utils.data_loader import (
#     load_dataset, get_class_distribution, split_train_test, load_test_set,
#     LABEL_COL, RANDOM_SEED, DATA_DIR, ORIGINAL_FILE,
# )

# print(f"Корень проекта: {PROJECT_ROOT}")
# print(f"Папка данных:   {DATA_DIR}")

# from src.utils.pipeline_config import load_pipeline_config

# # === GPU ПРОФИЛЬ — МЕНЯЙ ЗДЕСЬ ===
# GPU = "A100_40"  # варианты: "T4", "L4", "A100_40", "A100_80", "H100"
# pipeline_cfg = load_pipeline_config(GPU)

In [6]:
# Ставим зависимости из requirements.txt
!pip install -q -r {PROJECT_ROOT}/requirements.txt

___
## Предобработка данных

Очистка сырых данных: удаление дубликатов, повторов слов, циклов, обрезка приложений.

`data.json` → `data_after_eda.csv`

In [7]:
from src.utils.data_cleaner import run as run_cleaning

eda_path = DATA_DIR / "data_after_eda.csv"

if eda_path.exists():
    print(f"data_after_eda.csv уже существует ({len(pd.read_csv(eda_path))} записей), пропускаем")
else:
    run_cleaning()

data_after_eda.csv уже существует (1750 записей), пропускаем


___
## Разделение на train / test

Стратифицированное разбиение 80/20 с гарантией минимум 1 примера на класс в каждой части.
Аугментация применяется **только к train**. Оценка — **на test**.

In [8]:
from src.utils.data_loader import STAGE_FILES, TEST_FILE

train_path = DATA_DIR / STAGE_FILES[0]  # train_after_eda.csv
test_path = DATA_DIR / TEST_FILE         # data_test.csv

if train_path.exists() and test_path.exists():
    # Уже разбито — просто загружаем
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    print(f"Train/test уже существуют, загружены из файлов")
else:
    # Первый запуск — разбиваем оригинал
    original_path = DATA_DIR / ORIGINAL_FILE
    df_original = pd.read_csv(original_path)
    print(f"Загружен оригинал: {original_path.name} ({len(df_original)} записей)")
    df_train, df_test = split_train_test(df_original)

print(f"\nTrain: {len(df_train)} ({len(df_train)/(len(df_train)+len(df_test))*100:.0f}%)")
print(f"Test:  {len(df_test)} ({len(df_test)/(len(df_train)+len(df_test))*100:.0f}%)")

Train/test уже существуют, загружены из файлов

Train: 1409 (81%)
Test:  341 (19%)


In [9]:
# Распределение по классам в train и test
dist_train = get_class_distribution(df_train)
dist_test = get_class_distribution(df_test)

print(f"{'Класс':<70} {'Train':>6} {'Test':>5}")
print("-" * 85)
for cls in dist_train.index:
    tr = dist_train[cls]
    te = dist_test.get(cls, 0)
    print(f"  {cls:<68} {tr:>6} {te:>5}")
print("-" * 85)
print(f"  {'ИТОГО':<68} {len(df_train):>6} {len(df_test):>5}")

Класс                                                                   Train  Test
-------------------------------------------------------------------------------------
  Блок технического директора                                             197    49
  Блок директора по мощностям                                             193    48
  Блок директора по строительству                                         131    32
  Управление по проектным работам                                         108    27
  Блок заместителя генерального директора по безопасности                  99    24
  Генеральный директор                                                     82    20
  Проект "Нефтяные краюшки"                                                64    15
  Блок деректора по газу                                                   57    14
  Блок заместителя генерального директора по закупкам                      54    13
  Блок заместителя генерального директора по организационным вопросам     

___
## Baseline — классификация ДО аугментации

Обучаем те же модели на **оригинальном train** (без аугментации), 
чтобы потом сравнить с результатами после аугментации.

In [10]:
from src.classification.evaluate import load_data, evaluate_model
from src.classification.embeddings import prepare_features
from sklearn.preprocessing import LabelEncoder

# Загружаем оригинальный train (stage=0) и test
df_train_orig = load_dataset(stage=0)
df_test_baseline = load_test_set()

X_train_orig, y_train_orig_raw, X_test_orig, y_test_orig_raw = prepare_features(
    df_train_orig, df_test_baseline, use_cache=False
)

le_baseline = LabelEncoder()
y_train_orig = le_baseline.fit_transform(y_train_orig_raw)
y_test_orig = le_baseline.transform(y_test_orig_raw)
label_names_baseline = le_baseline.classes_

print(f"Train (без аугментации): {X_train_orig.shape}")
print(f"Test: {X_test_orig.shape}")
print(f"Классов: {len(label_names_baseline)}")

[Данные] Найден чекпоинт этапа 0: train_after_eda.csv (1409 записей)
[Данные] Тестовая выборка: data_test.csv (341 записей)
[TF-IDF] Параметры: max_features=50000, ngram_range=(1, 2)
[TF-IDF] Обучаю на 1409 текстах...
[TF-IDF] Готово: train (1409, 27985), test (341, 27985)
Train (без аугментации): (1409, 27985)
Test: (341, 27985)
Классов: 36


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

baseline_results = []

print("=" * 60)
print("BASELINE (без аугментации)")
print("=" * 60)

baseline_results.append(evaluate_model(
    name="[Baseline] Linear SVM",
    estimator=LinearSVC(max_iter=10000, random_state=RANDOM_SEED, dual="auto"),
    X_train=X_train_orig, y_train=y_train_orig,
    X_test=X_test_orig, y_test=y_test_orig,
    label_names=label_names_baseline,
    param_grid={"C": [0.01, 0.1, 1, 10]},
))

baseline_results.append(evaluate_model(
    name="[Baseline] Logistic Regression",
    estimator=LogisticRegression(solver="lbfgs", max_iter=1000, random_state=RANDOM_SEED),
    X_train=X_train_orig, y_train=y_train_orig,
    X_test=X_test_orig, y_test=y_test_orig,
    label_names=label_names_baseline,
    param_grid={"C": [0.01, 0.1, 1, 10]},
))

baseline_results.append(evaluate_model(
    name="[Baseline] Multinomial Naive Bayes",
    estimator=MultinomialNB(),
    X_train=X_train_orig, y_train=y_train_orig,
    X_test=X_test_orig, y_test=y_test_orig,
    label_names=label_names_baseline,
    param_grid={"alpha": [0.01, 0.1, 0.5, 1.0]},
))

In [ ]:
from src.classification.rubert_classifier import train_and_evaluate

# --- Настройки моделей ---
RUBERT_CONFIGS = [
    {
        "model_name": "cointegrated/rubert-tiny2",
        "short_name": "rubert-tiny2",
        "lr": 5e-4,
        "num_epochs": 15,
        "batch_size": 32,
    },
    {
        "model_name": "DeepPavlov/rubert-base-cased",
        "short_name": "rubert-base",
        "lr": 5e-5,
        "num_epochs": 15,
        "batch_size": 32,
    },
]

# for cfg in RUBERT_CONFIGS:
#     baseline_results.append(train_and_evaluate(
#         df_train=df_train_orig,
#         df_test=df_test_baseline,
#         model_name=cfg["model_name"],
#         lr=cfg["lr"],
#         num_epochs=cfg["num_epochs"],
#         batch_size=cfg["batch_size"],
#         name=f"[Baseline] {cfg['short_name']}",
#     ))


___
## Этап 1: LLM-генерация (< 15 -> 15)

Классы с менее чем 15 примерами дополняются новыми текстами,
сгенерированными через LLM.

In [11]:
# Путь до конфига модели 
# CONFIG_PATH = str(PROJECT_ROOT / "config_models" / "aug_configs" / "model_qwen.json")
# урезанная
# CONFIG_PATH = str(PROJECT_ROOT / "config_models" / "aug_configs" / "model_qwen_3b.json")
# квантизированная
# CONFIG_PATH = str(PROJECT_ROOT / "config_models" / "aug_configs" / "model_qwen_14b_unsloth.json")
# print(f"Конфиг модели: {CONFIG_PATH}")
# AWQ квантизиция
# CONFIG_PATH = str(PROJECT_ROOT / "config_models" / "aug_configs" / "model_vllm.json")
# print(f"Конфиг модели: {CONFIG_PATH}")

# # Если есть мощность A100 и лучше
CONFIG_PATH = str(PROJECT_ROOT / "config_models" / "aug_configs" / "model_vllm_32b.json")
print(f"Конфиг модели: {CONFIG_PATH}")

Конфиг модели: /content/VKR/code/config_models/aug_configs/model_vllm_32b.json


In [ ]:
from src.augmentation.stage1_llm_generate import run as run_stage1

run_stage1(CONFIG_PATH, pipeline_cfg=pipeline_cfg)

In [ ]:
df_after_s1 = load_dataset(stage=1)
dist_s1 = get_class_distribution(df_after_s1)
while (dist_s1 < 15).sum() != 0:# Проверяем результат после этапа 1
    print(f"Записей после этапа 1: {len(df_after_s1)}")
    print(f"Классов с < 15 примерами: {(dist_s1 < 15).sum()}")
    print(f"{"="*100}\n\n")
    print("Повторяем 1й этап")
    print(f"\n\n{"="*100}")
    run_stage1(CONFIG_PATH, pipeline_cfg=pipeline_cfg)
    df_after_s1 = load_dataset(stage=1)
    dist_s1 = get_class_distribution(df_after_s1)
print("Этап 1: Генерация с помощью LLM полностью завершен.")

## 3. Этап 2: ruT5-парафраз с чанкованием (< 35 -> 35)

Классы с менее чем 35 примерами пополняются через `fyaronskiy/ruT5-large-paraphraser`.
Длинные письма режутся на tokenizer-aware чанки и собираются обратно перед валидацией + LLM-судьёй.
Для ruT5 prompt-leak фильтр отключён, а фильтр длины считается относительно исходного письма.
Если после судьи классы не добраны, Stage 2 делает дополнительные циклы ruT5 → judge в этом же запуске.

In [12]:
from src.augmentation.stage2_paraphrase import run as run_stage2

run_stage2(CONFIG_PATH, pipeline_cfg=pipeline_cfg)

ЭТАП 2: Парафраз через LLM (< 35 → 35)
       источник парафраза: ТОЛЬКО ОРИГИНАЛЫ (stage 0)
[Данные] Чекпоинт этапа 2 не найден, загружен этап 1: data_after_stage1.csv (1533 записей)

[Этап 2] Классов для аугментации: 25
  «Блок бизнес-директора»: 15 → нужно ещё 20
  «Имущественные вопросы»: 15 → нужно ещё 20
  «Блок заместителя генерального директора по имуществу»: 15 → нужно ещё 20
  «Блок директора по персоналу»: 15 → нужно ещё 20
  «Блок исполнительного директора по реализации проекта "Большое месторождение"»: 15 → нужно ещё 20
  «Блок заместителя генерального директора по строительству»: 15 → нужно ещё 20
  «Подразделение по информационным технологиям»: 15 → нужно ещё 20
  «Проект "Южный"»: 15 → нужно ещё 20
  «Проект "Обустройство площадных объектов НГКМ Поменбше"»: 15 → нужно ещё 20
  «Проект "Восточный"»: 15 → нужно ещё 20
  «Проект «Обустройство Интересного лицензионного участка»»: 15 → нужно ещё 20
  «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: 15 → нужно ещё 20
  «П

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 05-04 10:58:37 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-04 10:58:37 [nixl_utils.py:34] NIXL is not available
WARNING 05-04 10:58:37 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-04 10:58:37 [model.py:555] Resolved architecture: Qwen2ForCausalLM
INFO 05-04 10:58:37 [model.py:1680] Using max model len 8192
INFO 05-04 10:58:37 [awq_marlin.py:256] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference
INFO 05-04 10:58:37 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  [Судья] Средняя оценка: 8.0, отсеяно < 5.0: 0, отобрано 4, средняя отобранных: 8.0
  [Раунд 1] После фильтров 8, после судьи 4, принято 4, всего 4/4
[Этап 2] Класс «Проект "Новая нефть"»: добавлено 4 парафразов
[Данные] Сохранён чекпоинт этапа 2: data_after_stage2.csv (1538 записей)
[Этап 2] Промежуточное сохранение: 2 классов обработано, 1538 записей

[Этап 2] Класс «Проект "Северная деревня"»: есть 29 (из них 29 оригиналов), источников парафраза: 29, нужно ещё 6
  [Раунд 1/5] Нужно ещё 6, генерируем батч из 31 парафразов


Rendering conversations:   0%|          | 0/31 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/31 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

KeyboardInterrupt: 

In [ ]:
# Проверяем результат после этапа 2
df_after_s2 = load_dataset(stage=2)
dist_s2 = get_class_distribution(df_after_s2)

print(f"Записей после этапа 2: {len(df_after_s2)}")
print(f"Классов с < 35 примерами: {(dist_s2 < 35).sum()}")

## 4. Этап 3: NLLB back-translation с чанкованием (< 50 -> 50)

Оставшиеся классы с менее чем 50 примерами дополняются через
обратный перевод RU → pivot → RU (`facebook/nllb-200-1.3B`) с tokenizer-aware чанкованием.

In [ ]:
from src.augmentation.stage3_back_translation import run as run_stage3

run_stage3(CONFIG_PATH, pipeline_cfg=pipeline_cfg)

In [ ]:
import gc
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import set_seed

from src.utils.data_loader import load_dataset, load_test_set
from src.classification.rubert_classifier import train_and_evaluate

# Если хочешь только rubert-tiny2, раскомментируй эту строку:
# RUN_CONFIGS = [cfg for cfg in RUBERT_CONFIGS if cfg["short_name"] == "rubert-tiny2"]

# Если хочешь прогнать все модели из RUBERT_CONFIGS: tiny + base
RUN_CONFIGS = RUBERT_CONFIGS

NUM_EPOCHS = 15
SEED = 42

df_test_stage_ablation = (
    df_test_baseline
    if "df_test_baseline" in globals()
    else load_test_set()
)

stage_datasets = [
    {"stage": "after_eda",    "stage_num": 0, "df": load_dataset(stage=0)},
    {"stage": "after_stage1", "stage_num": 1, "df": load_dataset(stage=1)},
    {"stage": "after_stage2", "stage_num": 2, "df": load_dataset(stage=2)},
    {"stage": "after_stage3", "stage_num": 3, "df": load_dataset(stage=3)},
]

rubert_stage_results = []

for cfg in RUN_CONFIGS:
    for stage_info in stage_datasets:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
        set_seed(SEED)

        stage_name = stage_info["stage"]
        df_train_stage = stage_info["df"]

        print("\n" + "=" * 90)
        print(f"MODEL: {cfg['short_name']} | STAGE: {stage_name} | TRAIN SIZE: {len(df_train_stage)}")
        print("=" * 90)

        result = train_and_evaluate(
            df_train=df_train_stage,
            df_test=df_test_stage_ablation,
            model_name=cfg["model_name"],
            lr=cfg["lr"],
            num_epochs=NUM_EPOCHS,
            batch_size=cfg["batch_size"],
            name=f"[{stage_name}] {cfg['short_name']}",
        )

        rubert_stage_results.append({
            "model": cfg["short_name"],
            "model_name": cfg["model_name"],
            "stage": stage_name,
            "stage_num": stage_info["stage_num"],
            "train_size": len(df_train_stage),
            "balanced_accuracy": float(result["balanced_accuracy"]),
            "macro_f1": float(result["macro_f1"]),
        })

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Сохраняем после каждого запуска, чтобы не потерять результаты при падении Colab
        tmp_df = pd.DataFrame(rubert_stage_results)
        tmp_path = Path(PROJECT_ROOT) / "results" / "rubert_stage_ablation_partial.csv"
        tmp_path.parent.mkdir(exist_ok=True)
        tmp_df.to_csv(tmp_path, index=False)

df_rubert_stages = pd.DataFrame(rubert_stage_results)
df_rubert_stages = df_rubert_stages.sort_values(["model", "stage_num"]).reset_index(drop=True)

# Дельты относительно предыдущего этапа
df_rubert_stages["delta_prev_balanced_accuracy"] = (
    df_rubert_stages.groupby("model")["balanced_accuracy"].diff()
)
df_rubert_stages["delta_prev_macro_f1"] = (
    df_rubert_stages.groupby("model")["macro_f1"].diff()
)

# Дельты относительно after_eda
eda_baseline = (
    df_rubert_stages[df_rubert_stages["stage"] == "after_eda"]
    .set_index("model")[["balanced_accuracy", "macro_f1"]]
)

df_rubert_stages["delta_eda_balanced_accuracy"] = df_rubert_stages.apply(
    lambda row: row["balanced_accuracy"] - eda_baseline.loc[row["model"], "balanced_accuracy"],
    axis=1,
)
df_rubert_stages["delta_eda_macro_f1"] = df_rubert_stages.apply(
    lambda row: row["macro_f1"] - eda_baseline.loc[row["model"], "macro_f1"],
    axis=1,
)

csv_path = Path(PROJECT_ROOT) / "results" / "rubert_stage_ablation.csv"
df_rubert_stages.to_csv(csv_path, index=False)

print("\nСохранено:", csv_path)
display(df_rubert_stages)


In [ ]:
# Проверяем финальный результат
df_final = load_dataset(stage=3)
dist_final = get_class_distribution(df_final)

print(f"Записей после всех этапов: {len(df_final)}")
print(f"Классов с < 50 примерами: {(dist_final < 50).sum()}")
print(f"\nМинимум примеров в классе: {dist_final.min()}")
print(f"Максимум примеров в классе: {dist_final.max()}")

In [ ]:
import matplotlib.pyplot as plt

# Сравниваем распределение: train до и после аугментации
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

dist_train.plot(kind="bar", ax=axes[0], color="salmon")
axes[0].set_title("Train до аугментации")
axes[0].set_ylabel("Количество примеров")
axes[0].tick_params(axis="x", rotation=45)

dist_final.plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("Train после аугментации")
axes[1].set_ylabel("Количество примеров")
axes[1].axhline(y=50, color="g", linestyle="--", alpha=0.5, label="Целевой минимум (50)")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

___
## Классификация — оценка качества аугментации

Обучаем на аугментированном train, оцениваем на отложенном test.
Данные загружаются один раз, все модели используют одни и те же эмбеддинги.

### Загрузка данных и эмбеддингов

In [ ]:
from src.classification.evaluate import load_data, evaluate_model

X_train, y_train, X_test, y_test, label_names = load_data()

print(f"Train: {X_train.shape}, Test: {X_test.shape}, Классов: {len(label_names)}")

### Linear SVM

In [ ]:
from sklearn.svm import LinearSVC

augmented_results = []

augmented_results.append(evaluate_model(
    name="[Augmented] Linear SVM",
    estimator=LinearSVC(max_iter=10000, random_state=RANDOM_SEED, dual="auto"),
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    label_names=label_names,
    param_grid={"C": [0.01, 0.1, 1, 10]},
))

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

augmented_results.append(evaluate_model(
    name="[Augmented] Logistic Regression",
    estimator=LogisticRegression(solver="lbfgs", max_iter=1000, random_state=RANDOM_SEED),
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    label_names=label_names,
    param_grid={"C": [0.01, 0.1, 1, 10]},
))

### Multinomial Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB

augmented_results.append(evaluate_model(
    name="[Augmented] Multinomial Naive Bayes",
    estimator=MultinomialNB(),
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    label_names=label_names,
    param_grid={"alpha": [0.01, 0.1, 0.5, 1.0]},
))

### BERT-модели (fine-tuning)


In [ ]:
from src.classification.rubert_classifier import train_and_evaluate

# df_final уже загружен выше (stage=3), df_test_baseline тоже
for cfg in RUBERT_CONFIGS:
    augmented_results.append(train_and_evaluate(
        df_train=df_final,
        df_test=df_test_baseline,
        model_name=cfg["model_name"],
        lr=cfg["lr"],
        num_epochs=10,
        batch_size=cfg["batch_size"],
        name=f"[Augmented] {cfg['short_name']}",
    ))


In [ ]:
from src.classification.rubert_classifier import train_and_evaluate

# df_final уже загружен выше (stage=3), df_test_baseline тоже
for cfg in RUBERT_CONFIGS:
    augmented_results.append(train_and_evaluate(
        df_train=df_final,
        df_test=df_test_baseline,
        model_name=cfg["model_name"],
        lr=cfg["lr"],
        num_epochs=10,
        batch_size=cfg["batch_size"],
        name=f"[Augmented] {cfg['short_name']}",
    ))

In [ ]:
from src.classification.rubert_classifier import train_and_evaluate

# df_final уже загружен выше (stage=3), df_test_baseline тоже
for cfg in RUBERT_CONFIGS:
    augmented_results.append(train_and_evaluate(
        df_train=df_final,
        df_test=df_test_baseline,
        model_name=cfg["model_name"],
        lr=cfg["lr"],
        num_epochs=10,
        batch_size=cfg["batch_size"],
        name=f"[Augmented] {cfg['short_name']}",
    ))


___
## Сравнение: Baseline vs Augmented

Сводная таблица и график — как изменились метрики после аугментации для каждого классификатора.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Названия моделей (без префикса [Baseline]/[Augmented])
model_names = [r["name"].split("] ")[1] for r in baseline_results]

metrics = ["balanced_accuracy", "macro_f1"]
metric_labels = ["Balanced Accuracy", "Macro F1"]

# Таблица
rows = []
print(f"{'Модель':<25} | {'Метрика':<20} | {'Baseline':>10} | {'Augmented':>10} | {'Delta':>10}")
print("-" * 83)
for i, name in enumerate(model_names):
    for m, ml in zip(metrics, metric_labels):
        b = baseline_results[i][m]
        a = augmented_results[i][m]
        delta = a - b
        sign = "+" if delta >= 0 else ""
        print(f"  {name:<23} | {ml:<20} | {b:>10.4f} | {a:>10.4f} | {sign}{delta:>9.4f}")
    print("-" * 83)
    rows.append({
        "stage": "baseline",
        "model": name,
        "balanced_accuracy": round(float(baseline_results[i]["balanced_accuracy"]), 4),
        "macro_f1": round(float(baseline_results[i]["macro_f1"]), 4),
    })
    rows.append({
        "stage": "augmented",
        "model": name,
        "balanced_accuracy": round(float(augmented_results[i]["balanced_accuracy"]), 4),
        "macro_f1": round(float(augmented_results[i]["macro_f1"]), 4),
    })

# График
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

x = np.arange(len(model_names))
width = 0.35

for ax, m, ml in zip(axes, metrics, metric_labels):
    vals_b = [r[m] for r in baseline_results]
    vals_a = [r[m] for r in augmented_results]

    bars_b = ax.bar(x - width/2, vals_b, width, label="Baseline", color="salmon")
    bars_a = ax.bar(x + width/2, vals_a, width, label="Augmented", color="steelblue")

    ax.set_title(ml)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=30, ha="right")
    ax.set_ylim(0, 1)
    ax.legend()

    # Подписи значений
    for bar in bars_b:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)
    for bar in bars_a:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("Baseline vs Augmented", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


results_dir = Path("/content/drive/MyDrive/VKR/code/results")
results_dir.mkdir(exist_ok=True)
pd.DataFrame(rows).to_csv(results_dir / "classification_results.csv", index=False)
print(f"\nСохранено: {results_dir / 'classification_results.csv'}")


In [ ]:
import pandas as pd
from pathlib import Path

# --- Сводим baseline и augmented результаты в один CSV ---
results_dir = Path(f"{PROJECT_ROOT}/results")
results_dir.mkdir(exist_ok=True)

rows = []
for r in baseline_results:
    rows.append({
        "stage": "baseline",
        "model": r["name"].replace("[Baseline] ", "").replace("[Augmented] ", ""),
        "balanced_accuracy": round(r["balanced_accuracy"], 4),
        "macro_f1": round(r["macro_f1"], 4),
    })
for r in augmented_results:
    rows.append({
        "stage": "augmented",
        "model": r["name"].replace("[Baseline] ", "").replace("[Augmented] ", ""),
        "balanced_accuracy": round(r["balanced_accuracy"], 4),
        "macro_f1": round(r["macro_f1"], 4),
    })

df_results = pd.DataFrame(rows)
csv_path = results_dir / "classification_results.csv"
df_results.to_csv(csv_path, index=False)
print(f"Сохранено: {csv_path}")
print()
print(df_results.to_string(index=False))
